# Klasterisasi dan Ekonometrika Panel-Spasial Pasar Tenaga Kerja Indonesia
### ACTION 2026 — Annual Competition of Data Science (UNESA)
**Subtema:** *“Turning Insight into Opportunity: Analisis Pengangguran Terbuka untuk Pemerataan Kesempatan Kerja”*  
**Unit Analisis:** 38 Provinsi di Indonesia (dengan panel seimbang 34 provinsi periode 2017–2024 = 272 observasi)  
**Sumber Data:** Badan Pusat Statistik (Sakernas 2017–2024 & Indikator Pasar Tenaga Kerja)

| Peran | Variabel | Makna dalam Pasar Tenaga Kerja | Satuan |
|---|---|---|:---:|
| **Y** | `TPT_Total` | Tingkat Pengangguran Terbuka (indikator utama penyerapan kerja) | % |
| **X1** | `Informal` | Proporsi pekerja sektor informal (penyangga subsisten) | % |
| **X2** | `log(Upah)` | Logaritma rata-rata upah bersih buruh/karyawan | log(Rp) |
| **X3** | `Educated_Mismatch` | Rasio mismatch pengangguran terdidik (SMK & PT vs SD) | rasio |

---


## 0. Setup Lingkungan & Dependensi


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RS = 42
np.random.seed(RS)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.bbox': 'tight', 'font.size': 9})

HIJAU = '#2F7D6E'
BATA  = '#C0392B'
PASIR = '#D9B26F'
ABU   = '#8896A0'

DATA_DIR = '../data/clean'
CHARTS_DIR = '../assets/charts'
print("Setup selesai. Siap membaca data resmi BPS.")


## 1. Memuat Data Panel & Georeferensi 38 Provinsi Indonesia


In [ ]:
master_38 = pd.read_csv(f'{DATA_DIR}/master_panel_provinsi_2024.csv')
panel_hist = pd.read_csv(f'{DATA_DIR}/panel_tpt_provinsi_2017_2024.csv')
coords = pd.read_csv(f'{DATA_DIR}/koordinat_provinsi_indonesia.csv')
dist_mat = pd.read_csv(f'{DATA_DIR}/jarak_km_provinsi_indonesia.csv', index_col=0)
W_knn4 = pd.read_csv(f'{DATA_DIR}/W_knn4_indonesia.csv', index_col=0)

df_prov = master_38[master_38["Provinsi"] != "Indonesia"].copy()
print(f"Total Provinsi Teranalisis : {len(df_prov)} Provinsi")
print(f"Total Panel Historis      : {len(panel_hist) - 1} Provinsi x 8 Tahun (2017-2024)")
display(coords.head(5))


## 2. Autokorelasi Spasial Global: Moran's I Lintasan Waktu (2017–2024)
Langkah pertama: menguji apakah pengangguran antar-provinsi di Indonesia bersifat acak atau mengelompok secara geografis.


In [ ]:
df_moran = pd.read_csv(f'{DATA_DIR}/moran_lintasan_tahun_tpt.csv')
display(df_moran)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(df_moran["tahun"], df_moran["moran_i"], marker="o", color=HIJAU, linewidth=2.5, markersize=8, label="Moran's I TPT")
ax.axhline(0, color=ABU, linestyle="--", alpha=0.7, label="Garis Netral (Acak Spasial)")
ax.set_title("Evolusi Autokorelasi Spasial Pengangguran Indonesia (2017–2024)\nMoran's I Global Antar-Provinsi", fontsize=11, fontweight="bold")
ax.set_xlabel("Tahun Pengamatan BPS Sakernas")
ax.set_ylabel("Nilai Indeks Moran's I")
ax.grid(True, linestyle=":", alpha=0.6)
for _, r in df_moran.iterrows():
    ax.annotate(f"{r['moran_i']:.3f}", (r['tahun'], r['moran_i']), textcoords="offset points", xytext=(0, 8), ha='center', fontweight="bold")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 3. Autokorelasi Spasial Lokal: LISA 2024 (Local Indicators of Spatial Association)
Mengidentifikasi kantong pengangguran regional (*Hotspot High-High*), wilayah agraris (*Coldspot Low-Low*), dan wilayah *shelter*.


In [ ]:
df_lisa = pd.read_csv(f'{DATA_DIR}/lisa_provinsi_2024.csv')
print("Distribusi Kuadran LISA 2024:")
display(df_lisa["kuadran"].value_counts())

fig, ax = plt.subplots(figsize=(8, 6))
colors_lisa = {
    "High-High (Hotspot Pengangguran)": BATA,
    "Low-Low (Coldspot / Agraris)": HIJAU,
    "Low-High (Shelter di Tengah Hotspot)": PASIR,
    "High-Low (Anomali Mandiri Tinggi)": "#6C5B9E"
}
for quad, col in colors_lisa.items():
    sub = df_lisa[df_lisa["kuadran"] == quad]
    ax.scatter(sub["z_TPT"], sub["lag_z_TPT"], color=col, s=70, alpha=0.85, label=quad, edgecolors="white", linewidth=0.5)

m_slope, b_intercept = np.polyfit(df_lisa["z_TPT"], df_lisa["lag_z_TPT"], 1)
x_line = np.linspace(df_lisa["z_TPT"].min(), df_lisa["z_TPT"].max(), 50)
ax.plot(x_line, m_slope * x_line + b_intercept, color='#243B3A', linewidth=1.8, label=f"Kemiringan Moran's I ({m_slope:.3f})")
ax.axvline(0, color=ABU, linestyle="--", alpha=0.5)
ax.axhline(0, color=ABU, linestyle="--", alpha=0.5)
ax.set_title("Moran Scatterplot & Klaster Spasial LISA (Agustus 2024)", fontsize=11, fontweight="bold")
ax.set_xlabel("Standar Deviasi TPT Provinsi (z)")
ax.set_ylabel("Spatial Lag Tetangga (Wz)")
ax.legend(loc="upper left", fontsize=8.5)
plt.tight_layout()
plt.show()


## 4. Pemodelan Ekonometrika Spasial: OLS vs SLX vs SAR vs SEM vs SDM
Membandingkan model standar non-spasial (OLS) dengan model yang mengakomodasi efek spasial (SLX, SAR, SEM, SDM).


In [ ]:
df_models = pd.read_csv(f'{DATA_DIR}/model_spasial_perbandingan.csv')
display(df_models)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(df_models["Model"], df_models["AIC"], color=[ABU, ABU, HIJAU, HIJAU, BATA], width=0.55)
ax.set_title("Perbandingan Kriteria Informasi Akaike (AIC) Antarmodel\n(Nilai Lebih Rendah Mengindikasikan Model Lebih Unggul)", fontsize=11, fontweight="bold")
ax.set_ylabel("Nilai AIC")
ax.set_xticklabels(["OLS", "SLX", "SAR", "SEM", "SDM\n(Terpilih)"])
ax.set_ylim(df_models["AIC"].min() - 15, df_models["AIC"].max() + 10)
for b in bars:
    h = b.get_height()
    ax.annotate(f"{h:.1f}", (b.get_x() + b.get_width() / 2, h), textcoords="offset points", xytext=(0, 4), ha='center', fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Dekomposisi Efek Spasial LeSage–Pace (SDM)
Menghitung efek langsung (*Direct Effect*), efek limpahan (*Spillover Effect*), dan efek total dari informalitas, upah, dan mismatch pendidikan.


In [ ]:
df_lesage = pd.read_csv(f'{DATA_DIR}/dekomposisi_efek_sdm_tpt.csv')
display(df_lesage)

x_indices = np.arange(len(df_lesage))
width = 0.28
fig, ax = plt.subplots(figsize=(9, 5))
rects1 = ax.bar(x_indices - width, df_lesage["Efek_Langsung (Direct)"], width, label="Efek Langsung (Domestik)", color=HIJAU)
rects2 = ax.bar(x_indices, df_lesage["Efek_Limpahan (Spillover / Indirect)"], width, label="Efek Limpahan (Spasial)", color=PASIR)
rects3 = ax.bar(x_indices + width, df_lesage["Efek_Total"], width, label="Efek Total", color=BATA)
ax.set_title("Dekomposisi Efek Spatial Durbin Model (LeSage–Pace)", fontsize=11, fontweight="bold")
ax.set_xticks(x_indices)
ax.set_xticklabels(["Informalitas (%)", "log(Upah Buruh)", "Mismatch Terdidik"])
ax.axhline(0, color='#243B3A', linestyle="-", linewidth=0.8)
ax.set_ylabel("Besaran Efek Marginal Terhadap TPT")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


## 6. Grid Search Klaster Multidimensi & Uji Stabilitas Subsampling Bootstrap


In [ ]:
df_grid = pd.read_csv(f'{DATA_DIR}/grid_konfigurasi_klaster.csv')
df_fitur = pd.read_csv(f'{DATA_DIR}/fitur_lintasan_provinsi.csv')
print(f"Total Konfigurasi Teruji : {len(df_grid)}")
print(f"Konfigurasi Lolos Syarat : {len(df_grid[df_grid['Layak']])}")
display(df_grid[df_grid["Layak"]].sort_values("Silhouette", ascending=False).head(5))
